# Bronze Exploration Notebook — Open Brewery DB

This notebook helps profile the **Bronze layer** and define the **Silver transformations**.
> Notebook to run after at least one successful Bronze extraction.


In [ ]:
%pip install pandas

from pathlib import Path
import json
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)


## 1) Bronze snapshot

In [ ]:
candidates = sorted(Path("../data/bronze").glob("data/bronze/breweries/ingestion_date=*/run_id=*/breweries.jsonl"))
BRONZE_FILE = candidates[-1] if candidates else None

if BRONZE_FILE is None or not BRONZE_FILE.exists():
    raise FileNotFoundError(
        "No Bronze JSONL file found. Set BRONZE_FILE manually or run the Bronze layer first."
    )
else:
    print(f"Using Bronze file: {BRONZE_FILE}")

In [ ]:
MANIFEST_FILE = BRONZE_FILE.parent / "manifest.json"
BRONZE_FILE, MANIFEST_FILE

## 2) Bronze data

In [ ]:
df = pd.read_json(BRONZE_FILE, lines=True)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
df.head(3)


In [ ]:
if MANIFEST_FILE.exists():
    manifest = json.loads(MANIFEST_FILE.read_text(encoding="utf-8"))
    manifest
else:
    print("Manifest file not found next to the Bronze JSONL.")


## 3) Basic schema profiling

In [ ]:
schema_df = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "non_null": [int(df[c].notna().sum()) for c in df.columns],
    "nulls": [int(df[c].isna().sum()) for c in df.columns],
    "null_pct": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
    "n_unique": [int(df[c].nunique(dropna=True)) for c in df.columns],
}).sort_values(["null_pct", "column"], ascending=[False, True])

schema_df


In [ ]:
# Focus on the dimensions most likely to be normalized in Silver
key_dims = [c for c in ["country", "state_province", "state", "city", "brewery_type"] if c in df.columns]
for col in key_dims:
    print(f"\n=== {col} ===")
    display(
        df[col]
        .astype("string")
        .str.strip()
        .fillna("<NULL>")
        .value_counts(dropna=False)
        .head(10)
        .to_frame("count")
    )


## 4) Brewery type domain

In [ ]:
observed_types = (
    df["brewery_type"].astype("string").str.strip().str.lower()
    if "brewery_type" in df.columns
    else pd.Series(dtype="string")
)

observed_type_counts = observed_types.fillna("<NULL>").value_counts(dropna=False).to_frame("count")
observed_type_counts


In [ ]:
documented_types = {
    "micro", "nano", "regional", "brewpub", "large",
    "planning", "bar", "contract", "proprietor", "closed"
}

if "brewery_type" in df.columns:
    observed = set(
        df["brewery_type"]
        .astype("string")
        .str.strip()
        .str.lower()
        .dropna()
        .unique()
        .tolist()
    )
    print("Observed but undocumented:", sorted(observed - documented_types))
    print("Documented but not observed:", sorted(documented_types - observed))


## 5) Numeric profiling for latitude/longitude

In [ ]:
for col in [c for c in ["latitude", "longitude"] if c in df.columns]:
    parsed = pd.to_numeric(df[col], errors="coerce")
    profile = {
        "original_non_null": int(df[col].notna().sum()),
        "parsed_non_null": int(parsed.notna().sum()),
        "failed_numeric_cast": int(df[col].notna().sum() - parsed.notna().sum()),
        "min": parsed.min(),
        "max": parsed.max(),
    }
    print(f"\n=== {col} ===")
    print(profile)


## 6) Duplicate and key profiling

In [ ]:
if "id" in df.columns:
    dup_count = int(df["id"].duplicated().sum())
    print("Duplicated ids:", dup_count)

    if dup_count:
        display(df[df["id"].duplicated(keep=False)].sort_values("id").head(20))
else:
    print("Column 'id' not found.")


## 7) Partitioning sanity check for Silver

In [ ]:
partition_cols = [c for c in ["country", "state_province"] if c in df.columns]
if len(partition_cols) == 2:
    part_stats = (
        df.groupby(partition_cols, dropna=False)
        .size()
        .reset_index(name="rows")
        .sort_values("rows", ascending=False)
    )
    print("Total partitions:", len(part_stats))
    display(part_stats.head(20))
else:
    print("Partition columns not available:", partition_cols)
